[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Digital_Communications.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Digital Communications

How bits become waveforms and survive the trip back: modulation, matched filtering, synchronization, and OFDM — a working QAM link and a working multicarrier link, both built from scratch in NumPy.

## 1. Pre-requisites

- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S4 (matched filters).
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 for the capacity backdrop.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Modulation: Bits → Waveforms* (~35 min)
**Goal:** map bits to constellation symbols; understand the energy/rate trade.
**Feeds into:** Session 2 (matched filter & eyes).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Modulation — Bits → Waveforms</b></summary>

**Timing (~35 min).** 10 min why a symbol is a complex number · 10 min constellations as packing · 8 min the demo · 7 min the capacity backdrop.

**Board first — where the complex number comes from.** $A\cos(2\pi f_c t + \phi)$ expands to $I\cos - Q\sin$: two *independent* knobs riding one carrier, because cosine and sine are orthogonal over a symbol period. So each symbol carries a complex number $I + jQ$, and the complex plane in these plots is physical rather than notational. Students often treat $I/Q$ as a mathematical convenience; it is two real signals on one carrier, and the orthogonality from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) is what makes it work.

**Then modulation as a packing problem.** Given a power budget — a disc of radius set by average energy — how many points can you place while keeping them far enough apart to survive noise? More points means more bits per symbol and less margin. That framing makes the whole design space obvious: QPSK has 4 points and 2 bits/symbol, 16-QAM has 16 and 4 bits/symbol, and the trade is visible before any simulation runs.

**Ask the room before running.** "Same SNR, twice the bits per symbol. What happens?" Then make it quantitative: with unit average energy, QPSK's points sit at radius 1 with nearest-neighbour distance $\sqrt2 \approx 1.41$; 16-QAM's minimum distance is about 0.63. That is roughly 7 dB of margin traded for 2 extra bits per symbol. Having the numbers turns "density costs margin" from a slogan into an exchange rate.

**Point at the normalisation, because it is what makes the comparison fair.** `pts / np.sqrt((np.abs(pts)**2).mean())` forces unit *average* energy, so both constellations spend the same power. Without it, 16-QAM would look worse purely because its raw points are larger, and the comparison would be meaningless. Fair comparisons need an explicit shared budget — worth naming as a general habit.

**Close with the referee.** Neither constellation is "better"; each is optimal in a range of SNR, and [Shannon capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) says how many bits/symbol a given SNR can support at all. This is exactly why real systems do *adaptive modulation* — your Wi-Fi link drops from 256-QAM to QPSK as you walk away from the router, moving down the constellation ladder as margin disappears. Ask the room to predict what happens to their throughput when they do that; the answer is visible in the two scatter plots.
</details>

## 2. Constellations

💡 **Intuition.** A passband waveform $A\cos(2\pi f_c t + \phi)$ has two independent knobs — amplitude-on-cosine and amplitude-on-sine — so each symbol period carries a **complex number** $I + jQ$. A *constellation* is the alphabet of complex points you allow: QPSK uses 4 (2 bits/symbol), 16-QAM uses 16 (4 bits/symbol). More points ⇒ more bits per symbol ⇒ points closer together ⇒ noise flips them more easily. Modulation design is packing points on a power budget — with [capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) as the referee.

In [2]:
def qam16():
    pts = np.array([a + 1j*b for a in (-3,-1,1,3) for b in (-3,-1,1,3)])
    return pts / np.sqrt((np.abs(pts)**2).mean())          # unit average energy

def qpsk():
    return np.exp(1j * (np.pi/4 + np.pi/2 * np.arange(4)))

n_sym = 3000
for name, const, snr_db in [("QPSK", qpsk(), 10), ("16-QAM", qam16(), 10)]:
    tx = const[rng.integers(0, len(const), n_sym)]
    noise = (rng.standard_normal(n_sym) + 1j*rng.standard_normal(n_sym)) / np.sqrt(2)
    rx = tx + noise * 10**(-snr_db/20)
    plt.scatter(rx.real, rx.imag, s=2, alpha=0.4, label=f"{name} @ {snr_db} dB")
plt.axis("equal"); plt.legend(); plt.title("Same SNR, two alphabets: density costs margin")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2038435/2927047130.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two constellations at the **same 10 dB SNR** and the same average transmit power. QPSK's four clusters are widely separated with clear gaps; 16-QAM's sixteen clusters are visibly crowded, with neighbouring clouds beginning to touch. Every point of contact is a symbol error waiting to happen.

**The exchange rate is computable.** Both constellations are normalised to unit average energy — that is what `pts / np.sqrt((np.abs(pts)**2).mean())` enforces, and it is what makes the comparison fair. Under that budget, QPSK's points sit at radius 1 with a nearest-neighbour distance of $\sqrt{2} \approx 1.41$; 16-QAM's minimum distance is about **0.63**. So doubling the bits per symbol from 2 to 4 cost roughly a factor of 2.2 in minimum distance, which is about **7 dB** of noise margin. That is the price list for density, and it is the entire content of the session.

**Where the complex plane comes from.** A passband waveform $A\cos(2\pi f_c t + \phi)$ expands as $I\cos - Q\sin$, and cosine and sine are orthogonal over a symbol period — the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) orthogonality from earlier in the track, earning its keep. So a carrier has two independent amplitude knobs, and each symbol genuinely carries a complex number. These axes are physical, not notational.

Seen that way, modulation is a **packing problem**: given a disc whose radius is set by your power budget, place as many points as possible while keeping them far enough apart to survive the noise. Everything else in constellation design follows from that.

**Neither alphabet is better — and that is why your Wi-Fi changes speed.** Each is optimal over a range of SNR, and [Shannon capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) sets how many bits per symbol a given SNR can support at all. Real links therefore use *adaptive modulation*: strong signal, use 256-QAM and enjoy the throughput; walk away from the router and the link steps down through 64-QAM, 16-QAM, QPSK, trading rate for margin as it goes. The two scatter plots above are two rungs of that ladder, and the router is choosing between them many times per second.

---
### 🕐 Session 2 of 4 — *Pulse Shaping, Matched Filters & Eye Diagrams* (~40 min)
**Goal:** put symbols on pulses without smearing neighbors; read link health from the eye.
**Builds on:** Session 1; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 3 (synchronization).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Pulse Shaping, Matched Filters & Eye Diagrams</b></summary>

**Timing (~40 min).** 10 min why pulses must overlap · 10 min the Nyquist criterion · 10 min the eye diagram as a diagnostic · 10 min the demo and its zero.

**Board first — set up the impossibility.** You cannot transmit a point; you transmit a *pulse* carrying it. A pulse confined to one symbol period is a rectangle, whose spectrum is a sinc with infinite bandwidth — unusable. Limiting bandwidth forces the pulse to spread in time, so it *must* overlap its neighbours. Ask how a receiver can possibly separate overlapping symbols. The answer is the elegant one: they may overlap everywhere except at the sampling instants.

**The Nyquist criterion, stated as a design requirement.** A raised-cosine pulse is 1 at its own sampling instant and exactly **zero** at every other symbol's. So at the moment you sample, every neighbour contributes nothing — zero inter-symbol interference — even though the waveform between samples is a tangle. Worth emphasising that this is a *condition on the pulse*, not on the channel, and it is why raised-cosine exists at all.

**Then the catch that motivates Session 3.** "No ISI *if you sample exactly on time*." Sample slightly off and every neighbour contributes something, because the zero crossings are only at the exact instants. That is the horizontal axis of the eye diagram, and it makes timing recovery a requirement rather than a refinement.

**Explain the RRC split — students find it mysterious.** The transmitter uses a *root*-raised-cosine and so does the receiver, because RRC ∗ RRC = raised cosine. Splitting the Nyquist pulse across both ends means the receive filter is *matched* to the transmit pulse (maximising SNR, per [Statistical SP](./Statistical_Signal_Processing.ipynb) S4) while the end-to-end response still satisfies the zero-ISI condition. Two requirements, one factorisation. That is why you never see plain raised-cosine at the transmitter alone.

**Teach reading the eye — it is the most practical skill here.** Vertical opening at the sampling instant is noise margin. Horizontal width of the opening is timing margin. Thickness of the traces is ISI plus noise. A closed eye means the link cannot work at any threshold. Engineers debug real links by looking at this picture before any numbers, and students should practise the same reflex.

**Handle the "0.0000" honestly — this is the important instruction.** The printed symbol error rate is zero, and that is *not* evidence of a good link, because the test cannot resolve one. Theoretical QPSK SER at 14 dB is about $5.4\times10^{-7}$, so 500 symbols would be expected to produce 0.00027 errors — observing zero is a certainty, not an achievement. You would need roughly **1.9 million symbols** to expect a single error. Ask the room what upper bound 500 clean symbols actually justifies: by the rule of three, about $3/500 = 0.006$, which is four orders of magnitude above the truth. Measuring rare events requires sample sizes scaled to their rarity, and a zero is usually a statement about the experiment rather than the system.
</details>

## 3. From Symbols to Samples

💡 **Intuition.** You can't transmit points — you transmit *pulses* carrying points, and bandwidth limits force pulses to be long, overlapping their neighbors. The escape is **Nyquist pulses** (raised cosine): they may overlap everywhere *except at the sampling instants*, where every other pulse crosses zero — no inter-symbol interference *if you sample exactly on time*. The receiver applies the [matched filter](./Statistical_Signal_Processing.ipynb) for SNR, and the **eye diagram** — all symbol periods overlaid — shows your margins: vertical opening = noise margin, horizontal = timing margin.

In [3]:
# Root-raised-cosine link, oversampled 8x
sps, beta, span = 8, 0.35, 8
t_rrc = np.arange(-span*sps, span*sps + 1) / sps
def rrc(t, beta):
    out = np.zeros_like(t, float)
    for i, ti in enumerate(t):
        if abs(ti) < 1e-9: out[i] = 1 - beta + 4*beta/np.pi
        elif abs(abs(4*beta*ti) - 1) < 1e-9:
            out[i] = beta/np.sqrt(2)*((1+2/np.pi)*np.sin(np.pi/(4*beta)) + (1-2/np.pi)*np.cos(np.pi/(4*beta)))
        else:
            out[i] = (np.sin(np.pi*ti*(1-beta)) + 4*beta*ti*np.cos(np.pi*ti*(1+beta))) / (np.pi*ti*(1-(4*beta*ti)**2))
    return out / np.sqrt((out**2).sum())
h_rrc = rrc(t_rrc, beta)

syms = qpsk()[rng.integers(0, 4, 500)]
up = np.zeros(len(syms)*sps, complex); up[::sps] = syms
tx_wave = np.convolve(up, h_rrc)

snr_db = 14
noise = (rng.standard_normal(len(tx_wave)) + 1j*rng.standard_normal(len(tx_wave))) / np.sqrt(2)
rx_wave = tx_wave + noise * 10**(-snr_db/20) / np.sqrt(sps)
mf_out = np.convolve(rx_wave, h_rrc)                      # matched filter (RRC ∗ RRC = raised cosine)

delay = len(h_rrc) - 1
plt.figure(figsize=(8, 3))
for k in range(60, 260):                                   # overlay 2-symbol windows: the EYE
    seg = mf_out[delay + k*sps - sps : delay + k*sps + sps]
    plt.plot(np.arange(-sps, sps)/sps, seg.real, "C0", alpha=0.12, linewidth=0.8)
plt.axvline(0, color="r", linestyle=":", linewidth=1)
plt.title("Eye diagram @ 14 dB: open eye = healthy link; sample at the red line")
plt.xlabel("time [symbols]"); plt.tight_layout(); plt.show()

rx_syms = mf_out[delay::sps][:len(syms)]
decided = qpsk()[np.argmin(np.abs(rx_syms[:, None] - qpsk()[None, :]), axis=1)]
print(f"symbol error rate at 14 dB: {(decided != syms).mean():.4f} over {len(syms)} symbols")

symbol error rate at 14 dB: 0.0000 over 500 symbols


/tmp/ipykernel_2038435/1375707096.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("time [symbols]"); plt.tight_layout(); plt.show()


**What just happened.** A wide-open eye and a printed symbol error rate of **0.0000 over 500 symbols**. The eye is the real result; the zero is not a measurement, and it is worth being precise about why.

**Start with the number, because it is a trap.** Theoretical QPSK symbol error rate at 14 dB is about $5.4\times10^{-7}$. Across 500 symbols that predicts $2.7\times10^{-4}$ errors — so observing zero was essentially guaranteed before the cell ran, and it would have been equally guaranteed for a link 100× worse. To expect even one error you would need roughly **1.9 million symbols**. By the rule of three, zero errors in 500 trials justifies only a 95% upper bound of $3/500 = 0.006$, which sits four orders of magnitude above the true rate.

So "0.0000" reports the *experiment's* resolution, not the link's quality. Measuring a rare event requires a sample size scaled to its rarity, and BER simulations in practice run millions of symbols or use importance sampling for exactly this reason. A zero in an error-rate column should always prompt "how many trials?" before it prompts satisfaction.

**Now the eye, which does carry information.** Read it in three parts. The **vertical** opening at the red line is noise margin — how much noise can be added before traces cross the decision threshold. The **horizontal** width of the opening is timing margin — how far off the ideal instant you can sample and still decide correctly. And the **thickness** of the traces is residual ISI plus noise. A single picture, three diagnostics, which is why engineers look at an eye before they look at any number.

**Why the pulses can overlap and still work.** You cannot transmit a point, only a pulse carrying one, and bandwidth limits force pulses to be long — so they *do* overlap their neighbours, visibly, between the sampling instants. The escape is the **Nyquist criterion**: a raised-cosine pulse is 1 at its own sampling instant and exactly zero at every other symbol's. At the moment you sample, every neighbour contributes nothing. Zero ISI is engineered into the pulse shape rather than removed by the receiver.

**And the qualifier is the whole of Session 3.** No ISI *if you sample exactly on time*. Off-timing loses the zero crossings and every neighbour starts leaking in — which is precisely what the horizontal eye opening measures. Timing recovery is therefore a requirement, not a refinement.

**One design detail worth noticing.** Both ends use a *root*-raised-cosine, because RRC ∗ RRC = raised cosine. Splitting the Nyquist pulse across transmitter and receiver means the receive filter is *matched* to the transmitted pulse — maximising SNR by [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 — while the end-to-end response still satisfies the zero-ISI condition. Two requirements satisfied by one factorisation, which is why you never see plain raised-cosine at the transmitter alone.

---
### 🕐 Session 3 of 4 — *Synchronization* (~35 min)
**Goal:** find the frame and the phase: correlation sync and the cost of being wrong.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (OFDM).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Synchronization</b></summary>

**Timing (~35 min).** 8 min what the receiver does not know · 10 min the preamble correlation · 10 min the two-syncs-from-one-peak idea · 7 min the demo and its honesty.

**Board first — enumerate the receiver's ignorance.** It does not know when the frame starts, what the carrier phase is, or (in general) the exact symbol rate. Session 2's zero-ISI guarantee was conditional on sampling *exactly on time* — so without synchronisation, none of it works. Frame this session as the one that makes the previous one usable, rather than as a separate topic.

**The preamble correlation is the matched filter again — say so explicitly.** Prepend a known sequence, slide it across the stream, and the peak marks the frame start. That is detection in *time*, using the same theorem as [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 and the same operation as radar [pulse compression](./Radar_Signal_Processing.ipynb). The matched filter has now appeared in three workshops doing three apparently different jobs; pointing that out is worth more than any individual application.

**The elegant part: one correlation, two syncs.** The correlation *magnitude* peaks at the right position — that is timing. The correlation *phase* at that peak is the carrier phase offset — because correlating a rotated signal against an unrotated template produces a complex value carrying exactly that rotation. Have the room work out why before you tell them: $\sum (s_k e^{j\phi})\overline{s_k} = e^{j\phi}\sum|s_k|^2$, so the phase falls straight out. Free information from a computation you were doing anyway.

**Ask the room.** "Why must the preamble be a specific known sequence rather than any convenient pattern?" Because it needs a sharp autocorrelation — a strong peak at zero lag and small values everywhere else — or the peak is ambiguous and sync lands in the wrong place. This is why real standards use carefully designed sequences (Barker, Zadoff–Chu, m-sequences) rather than arbitrary bits. Ask what a preamble of all-identical symbols would do; its autocorrelation is broad and flat, and timing becomes unrecoverable.

**Note the honest caveat about the demo's conditions.** The channel here applies a constant phase offset and nothing else — no frequency offset, no timing drift, no multipath. Real receivers face a carrier *frequency* error (from oscillator mismatch and Doppler) that makes the phase rotate continuously, so a single phase estimate goes stale within a few symbols. That is why real systems run a phase-locked loop after acquisition rather than a one-shot correction. Say this, or students will think synchronisation is a solved one-liner.

**Point at the vestigial line.** `est_phase = ... if False else np.angle(peak)` contains a dead branch — the `if False` disables an alternative normalisation. It is harmless (a scale factor does not change the angle) but it is leftover code, and it is fair to say so rather than pretend it is meaningful.
</details>

## 4. Where Does the Frame Start?

💡 **Intuition.** The receiver knows neither *when* symbols start nor *what phase* the oscillator drifted to. Both are solved with correlation: prepend a known **preamble**; the receiver slides it across the incoming stream, and the correlation peak marks the frame start ([matched filter](./Statistical_Signal_Processing.ipynb) again — detection in time). The *phase* of that same peak reveals the carrier phase offset — one correlation, two syncs. Every Wi-Fi packet begins exactly this way.

In [4]:
preamble = qpsk()[rng.integers(0, 4, 64)]
payload = qpsk()[rng.integers(0, 4, 400)]
frame = np.concatenate([preamble, payload])

phase_off = 0.6                                          # unknown carrier phase [rad]
start = 137                                              # unknown start position
stream = np.concatenate([ (rng.standard_normal(start)+1j*rng.standard_normal(start))*0.2,
                          frame * np.exp(1j*phase_off),
                          (rng.standard_normal(200)+1j*rng.standard_normal(200))*0.2 ])
stream += (rng.standard_normal(len(stream)) + 1j*rng.standard_normal(len(stream))) * 0.1

corr = np.abs(np.correlate(stream, preamble, "valid"))
est_start = int(np.argmax(corr))
peak = np.correlate(stream, preamble, "valid")[est_start]
est_phase = np.angle(peak / (np.abs(preamble)**2).sum() * len(preamble)) if False else np.angle(peak)

plt.figure(figsize=(8, 2.4))
plt.plot(corr); plt.axvline(start, color="r", linestyle=":", label=f"true start {start}")
plt.legend(); plt.title(f"preamble correlation: peak at {est_start}, phase estimate {est_phase:.3f} rad (true {phase_off})")
plt.tight_layout(); plt.show()

fixed = stream[est_start + 64 : est_start + 64 + 400] * np.exp(-1j * est_phase)
decided = qpsk()[np.argmin(np.abs(fixed[:, None] - qpsk()[None, :]), axis=1)]
print(f"payload symbol errors after sync: {(decided != payload).sum()} / 400")

payload symbol errors after sync: 0 / 400


/tmp/ipykernel_2038435/234182044.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The receiver was handed a stream containing noise, then a frame at an unknown offset with an unknown phase rotation, then more noise. One correlation recovered **both** unknowns, and the payload decoded with **0 errors out of 400**.

**One correlation, two syncs — this is the elegant part.** The correlation *magnitude* peaks where the preamble aligns, which gives timing. The correlation *phase* at that peak gives the carrier phase offset, because
$$\sum_k (s_k e^{j\phi})\,\overline{s_k} = e^{j\phi}\sum_k |s_k|^2,$$
so rotating the whole frame by $\phi$ rotates the correlation peak by exactly $\phi$ and leaves the magnitude untouched. The phase estimate is free — it comes from a computation you were already performing for timing.

And the operation itself is one this curriculum keeps reusing: sliding a known template across data and taking the peak is the **matched filter** from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4, the same operation as radar [pulse compression](./Radar_Signal_Processing.ipynb). Three workshops, three apparently unrelated jobs, one theorem.

**Why the preamble must be designed, not arbitrary.** It needs a sharp autocorrelation — a strong peak at zero lag and small values elsewhere — or the peak is ambiguous and sync lands in the wrong place. A preamble of identical repeated symbols would have a broad, flat autocorrelation and timing would be unrecoverable. This is why real standards specify particular sequences (Barker, Zadoff–Chu, m-sequences) rather than convenient bit patterns, and why every Wi-Fi packet begins with a specific preamble rather than with data.

**Now the honest limits of this demo.** The channel applies a *constant* phase offset and nothing else. Real receivers face a carrier **frequency** error — oscillator mismatch and Doppler — which makes the phase rotate continuously, so a single estimate goes stale within a few symbols and the constellation visibly spins. That is why production receivers run a phase-locked loop after acquisition rather than a one-shot correction, and why frequency offset estimation typically comes before phase correction. There is also no timing drift here (sample clocks are assumed identical) and no multipath, which Session 4 addresses.

So read "0 / 400 errors" as confirming the *principle* under favourable conditions, not as a claim that synchronisation is a solved one-liner. It is the hardest part of a real receiver, and this cell shows the first step of it.

One small note: `est_phase = ... if False else np.angle(peak)` carries a disabled branch. The alternative normalisation would only have rescaled the peak, and a positive scale factor does not change an angle — so the two are equivalent and the dead code is harmless leftover.

---
### 🕐 Session 4 of 4 — *OFDM in 40 Minutes* (~40 min)
**Goal:** beat multipath by going wide-and-slow: the FFT as a modem.
**Builds on:** Session 3; [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S7.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: OFDM in 40 Minutes</b></summary>

**Timing (~40 min).** 8 min the multipath problem · 10 min wide-and-slow · 12 min the cyclic prefix, which is the whole trick · 10 min the demo.

**Board first — make multipath concrete.** An echo arriving 1 µs late is 300 m of extra path, entirely ordinary indoors. If your symbol lasts 0.1 µs, that echo lands ten symbols later and smears across all of them — inter-symbol interference on a scale no equaliser handles cheaply. Now ask the fix. The instinct is "build a better equaliser"; OFDM's answer is to change the problem instead.

**Wide-and-slow, stated as the judo move.** Rather than one fast stream, send $N$ slow ones in parallel on separate subcarriers. Each symbol is now $N$ times longer, so a 1 µs echo is a small fraction of a symbol rather than ten of them. The total data rate is unchanged — you traded speed per stream for number of streams — and the channel went from hostile to benign without any equaliser improving.

**Then the FFT, which is what makes it affordable.** $N$ separate oscillators and $N$ separate modulators would be absurd hardware. But the subcarriers are orthogonal complex exponentials at multiples of a fundamental — which is exactly the DFT basis. So the IFFT *is* the modulator and the FFT *is* the demodulator. Say plainly that OFDM only became practical because the FFT made it $O(N\log N)$; this is [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb)'s algorithm as the enabling technology for modern wireless.

**The cyclic prefix is the cleverest idea in the workshop — give it real time.** A channel performs **linear** convolution, but the FFT diagonalises **circular** convolution. Copying the last `cp` samples to the front makes the block *look* periodic to the channel over the observation window, so linear convolution acts as circular convolution — and then, by the convolution theorem, the channel becomes a single complex multiply per subcarrier. Equalisation collapses to `Y / H`: one division per subcarrier, no filter design, no adaptive algorithm. Ask the room to compare that against equalising a wideband single-carrier channel, which needs a long adaptive filter and never stops working.

**Ask the room.** "The cyclic prefix is redundant data — pure overhead. Why is it worth it?" Because it converts an intractable equalisation problem into a division. Here 16 samples of prefix on 64 of data is 25% overhead, a real cost that buys away an entire subsystem. Then the design constraint: the prefix must be *longer than the channel's delay spread*, or the trick fails and inter-block interference returns. That single inequality sets the prefix length in every real standard.

**Be honest about what the demo assumes.** `H = np.fft.fft(channel, Nfft)` uses the *true* channel — perfect channel state information. Real receivers estimate $H$ from pilot subcarriers, and estimation error degrades performance. Also absent: frequency offset (to which OFDM is notoriously sensitive, because it destroys subcarrier orthogonality and causes inter-carrier interference) and the high peak-to-average power ratio that makes OFDM hard on amplifiers. Mention PAPR — it is the main practical complaint about OFDM and the reason 4G uplinks use SC-FDMA instead.
</details>

## 5. OFDM

💡 **Intuition.** Multipath (echoes) smears fast single-carrier symbols into each other. OFDM's judo move: send **many slow streams in parallel**, one per subcarrier — and use the IFFT to pack them, the FFT to unpack. The **cyclic prefix** turns the channel's *linear* convolution into *circular* convolution ([Foundations 1 S8](./Foundations_of_Signal_Processing_1.ipynb)!), so the whole channel collapses to one complex multiply per subcarrier — equalization becomes division. Wi-Fi, LTE/5G, and DVB are exactly this.

In [5]:
Nfft, cp, n_ofdm = 64, 16, 200
channel = np.array([1.0, 0, 0.5, 0, 0, 0.3j])            # nasty multipath

data = qpsk()[rng.integers(0, 4, (n_ofdm, Nfft))]
tx = []
for row in data:
    sym = np.fft.ifft(row) * np.sqrt(Nfft)
    tx.append(np.concatenate([sym[-cp:], sym]))           # cyclic prefix
tx = np.concatenate(tx)

rx = np.convolve(tx, channel)[:len(tx)]
rx += (rng.standard_normal(len(rx)) + 1j*rng.standard_normal(len(rx))) * 0.05

H = np.fft.fft(channel, Nfft)                             # channel per subcarrier (known/est. via pilots)
eq_syms, raw_syms = [], []
for k in range(n_ofdm):
    blk = rx[k*(Nfft+cp)+cp : (k+1)*(Nfft+cp)]
    Y = np.fft.fft(blk) / np.sqrt(Nfft)
    raw_syms.append(Y); eq_syms.append(Y / H)             # equalization = one division!
raw = np.concatenate(raw_syms); eq = np.concatenate(eq_syms)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].scatter(raw.real, raw.imag, s=1, alpha=0.3); axes[0].set_title("before equalization: smeared by multipath")
axes[1].scatter(eq.real, eq.imag, s=1, alpha=0.3); axes[1].set_title("after Y/H: constellation restored")
for ax in axes: ax.set_aspect("equal")
plt.tight_layout(); plt.show()

decided = qpsk()[np.argmin(np.abs(eq.ravel()[:, None] - qpsk()[None, :]), axis=1)]
print(f"OFDM symbol error rate through multipath: {(decided != data.ravel()).mean():.4f}")

OFDM symbol error rate through multipath: 0.0002


/tmp/ipykernel_2038435/1503258650.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The left panel is a smear — a multipath channel with echoes at 3 and 6 sample delays has scrambled the constellation past recognition. The right panel is four clean QPSK clusters. The operation between them is `Y / H`: **one complex division per subcarrier**. Symbol error rate through that hostile channel, **0.0002** — about 3 errors in 12800 symbols.

**Equalisation became division, and that is the whole point.** A wideband single-carrier link through this channel would need a long adaptive equaliser — many taps, a convergence period, ongoing tracking, and real complexity. OFDM replaces all of it with a scalar divide per subcarrier. The channel did not become easier; the *representation* changed so that the channel is diagonal in it.

**The cyclic prefix is what makes that legal.** A physical channel performs **linear** convolution, but the FFT diagonalises **circular** convolution — those are different operations, and the difference is exactly the edge effects. Copying the last 16 samples to the front makes each block look periodic to the channel across the observation window, so linear convolution *acts as* circular convolution over the part we keep. Then the convolution theorem applies and the channel collapses to one multiply per subcarrier. This is [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb)'s circular-vs-linear convolution distinction — usually a pedantic footnote — turned into the enabling trick of modern wireless.

The prefix is pure overhead: 16 samples of redundancy for every 64 of data, **25%** of the airtime, discarded on arrival. It is worth it because it deletes an entire subsystem. And it sets a hard design rule — the prefix must be **longer than the channel's delay spread**, or inter-block interference returns and the whole construction fails. That single inequality determines prefix length in Wi-Fi, LTE, and DVB.

**Why parallel-and-slow beats fast-and-equalised.** An echo 1 µs late is 300 m of extra path, entirely ordinary indoors. Against a 0.1 µs symbol that echo lands ten symbols later and smears across all of them. Make each symbol $N$ times longer by sending $N$ streams in parallel, and the same echo becomes a small fraction of one symbol. Total rate is unchanged — speed per stream traded for number of streams — and the channel becomes benign without any equaliser getting better.

The FFT is what makes it affordable: the subcarriers are orthogonal complex exponentials at multiples of a fundamental, which is precisely the DFT basis, so the IFFT *is* the modulator. $N$ oscillators would be absurd hardware; $O(N\log N)$ is a chip.

**And the honest caveats.** `H = np.fft.fft(channel, Nfft)` uses the **true** channel — perfect channel state information, which no receiver has. Real systems estimate $H$ from pilot subcarriers and pay for the estimation error. Two further omissions matter in practice: carrier **frequency offset**, to which OFDM is notoriously sensitive because it breaks subcarrier orthogonality and produces inter-carrier interference; and the high **peak-to-average power ratio** of a sum of many subcarriers, which forces amplifiers to back off and is the main practical complaint about OFDM — enough so that LTE uplinks use SC-FDMA instead. The mechanism here is real and complete; the engineering around it is where the difficulty lives.

## 6. Conclusion

Bits ride complex symbols; Nyquist pulses dodge ISI; one correlation finds both time and phase; and OFDM turns a hostile channel into $N$ trivial ones via the FFT. You've built every layer of a real modem below the error-correcting code.

---
## Where next

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the capacity these designs chase.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — run this against *real* airwaves.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate front-ends around every modem.